# Zain Jordan Customer 360 AI Workshop  
## Class 1: Getting Started with the Telecom Customer 360 Database

### Workshop Context

This notebook is designed for a beginner-friendly AI workshop for **Zain Jordan**, a telecom company.

In this first class, we will learn how to:

1. Understand the Zain Jordan Customer 360 database.
2. Connect Python to a SQLite database.
3. Explore tables and columns.
4. Run simple SQL queries.
5. Analyze telecom business questions.
6. Create reusable Python functions.
7. Prepare the foundation for LangChain tools, SQL agents, RAG, multi-agent systems, MCP, and capstone projects.

---

### Big Idea

We are not learning Python randomly.

We are learning how to build AI applications on top of enterprise telecom data.

This same database will later be used for:

- SQL chatbot
- LangChain tools
- LangChain SQL agent
- RAG from database rows
- Multi-agent customer care copilot
- MCP tools
- Team capstone projects


# 1. Dataset Explanation

This database is a synthetic telecom **Customer 360** database.

Customer 360 means we connect many parts of the customer journey:

- Customer profile
- Account information
- Subscriptions
- Plans and add-ons
- Billing and payments
- Data, call, SMS, and roaming usage
- Support interactions
- Complaints
- Customer satisfaction
- Churn risk
- Customer value
- Campaigns
- Network towers and network events

The goal is to help participants understand how telecom companies can use AI agents to answer business questions and recommend actions.


# 2. Class 1 Learning Outcomes

By the end of this notebook, you should be able to:

1. Connect to the Zain Jordan SQLite database.
2. List all tables in the database.
3. Preview important telecom tables.
4. Run simple SQL queries using Python.
5. Use joins to combine customer, subscription, and plan information.
6. Analyze churn risk, complaints, network events, and campaigns.
7. Create reusable Python functions.
8. Build a simple Customer 360 view for one customer.


# 3. Install and Import Libraries

For Class 1, we only need Python's built-in `sqlite3` library and `pandas`.

`sqlite3` helps us connect to the database.  
`pandas` helps us display query results as tables.


In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path
import os

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

print("Libraries imported successfully.")


# 4. Upload or Locate the Database File

If you are using Google Colab:

1. Run the next cell.
2. Upload the file: `zain_customer_360_ai_demo.db`
3. Continue with the notebook.

If you are running locally, place the database file in the same folder as this notebook.


In [ ]:
# Run this cell only if you are using Google Colab.
# If you are running locally, you can skip this cell.

try:
    from google.colab import files
    uploaded = files.upload()
    print("Uploaded files:", list(uploaded.keys()))
except Exception:
    print("Google Colab upload is not available in this environment.")
    print("If running locally, make sure the .db file is in the same folder as this notebook.")


# 5. Set the Database Path

This cell tries to automatically find a SQLite `.db` file in the current folder.

If your database has a different file name, update `DB_PATH` manually.


In [ ]:
# Try to automatically detect the database file
db_files = [file for file in os.listdir() if file.endswith(".db")]

if db_files:
    DB_PATH = db_files[0]
else:
    DB_PATH = "zain_customer_360_ai_demo.db"

print("Database path:", DB_PATH)
print("File exists:", Path(DB_PATH).exists())


# 6. Connect to the SQLite Database

The connection object is the bridge between Python and the database.


In [ ]:
conn = sqlite3.connect(DB_PATH)

print("Connected to database successfully.")


# 7. List All Tables

SQLite stores information about tables in a system table called `sqlite_master`.

This query shows all tables in our database.


In [ ]:
query = '''
SELECT name 
FROM sqlite_master 
WHERE type = 'table'
ORDER BY name;
'''

tables_df = pd.read_sql_query(query, conn)
tables_df


# 8. Count the Number of Tables


In [ ]:
number_of_tables = len(tables_df)

print(f"The database has {number_of_tables} tables.")


# 9. Show Row Counts for All Tables

This helps us understand which tables are large and which tables are small.

For example:

- `plans` is small because companies have limited plan options.
- `data_usage_sessions` is large because customers generate many internet usage records.


In [ ]:
table_counts = []

for table_name in tables_df["name"]:
    count_query = f"SELECT COUNT(*) AS row_count FROM {table_name}"
    count = pd.read_sql_query(count_query, conn)["row_count"][0]
    table_counts.append({
        "table_name": table_name,
        "row_count": count
    })

table_counts_df = pd.DataFrame(table_counts)
table_counts_df.sort_values("row_count", ascending=False)


# 10. Preview the Customers Table

The `customers` table contains customer profile information.


In [ ]:
customers_df = pd.read_sql_query('''
SELECT * 
FROM customers 
LIMIT 5;
''', conn)

customers_df


# 11. Understand Columns in a Table

`PRAGMA table_info(table_name)` shows the structure of a SQLite table.


In [ ]:
pd.read_sql_query("PRAGMA table_info(customers);", conn)


# 12. Check the Churn Score Table Structure


In [ ]:
pd.read_sql_query("PRAGMA table_info(customer_churn_scores);", conn)


# 13. Business Question 1: Customers by City

Question:

**Which cities have the most customers?**


In [ ]:
query = '''
SELECT 
    city,
    COUNT(*) AS total_customers
FROM customers
GROUP BY city
ORDER BY total_customers DESC;
'''

customers_by_city = pd.read_sql_query(query, conn)
customers_by_city


# 14. Business Question 2: Customers by Segment

Question:

**What types of customers does Zain Jordan have?**


In [ ]:
query = '''
SELECT 
    customer_segment,
    COUNT(*) AS total_customers
FROM customers
GROUP BY customer_segment
ORDER BY total_customers DESC;
'''

customers_by_segment = pd.read_sql_query(query, conn)
customers_by_segment


# 15. Business Question 3: Active vs Inactive Customers


In [ ]:
query = '''
SELECT 
    status,
    COUNT(*) AS total_customers
FROM customers
GROUP BY status;
'''

customer_status = pd.read_sql_query(query, conn)
customer_status


# 16. Explore Telecom Plans

The `plans` table contains mobile, internet, prepaid, postpaid, and business plans.


In [ ]:
plans_df = pd.read_sql_query('''
SELECT 
    plan_id,
    plan_name,
    plan_category,
    service_type,
    monthly_fee_jod,
    data_allowance_gb,
    local_minutes,
    contract_months
FROM plans
ORDER BY monthly_fee_jod DESC;
''', conn)

plans_df


# 17. Business Question 4: Top Expensive Plans


In [ ]:
query = '''
SELECT 
    plan_name,
    plan_category,
    service_type,
    monthly_fee_jod,
    data_allowance_gb,
    local_minutes
FROM plans
ORDER BY monthly_fee_jod DESC
LIMIT 5;
'''

top_plans = pd.read_sql_query(query, conn)
top_plans


# 18. First Important Join: Customers + Subscriptions + Plans

Real business questions often need multiple tables.

Here:

- Customer information is in `customers`
- Subscription information is in `subscriptions`
- Plan information is in `plans`

We use `JOIN` to connect them.


In [ ]:
query = '''
SELECT 
    c.customer_id,
    c.full_name,
    c.city,
    c.customer_segment,
    s.subscription_id,
    s.msisdn,
    s.service_type,
    p.plan_name,
    p.monthly_fee_jod,
    p.data_allowance_gb
FROM customers c
JOIN subscriptions s 
    ON c.customer_id = s.customer_id
JOIN plans p 
    ON s.plan_id = p.plan_id
LIMIT 10;
'''

customer_plan_df = pd.read_sql_query(query, conn)
customer_plan_df


# 19. Business Question 5: Churn Risk Summary

Churn means the customer may leave the company.

Question:

**How many customers are low, medium, or high churn risk?**


In [ ]:
query = '''
SELECT 
    risk_level,
    COUNT(*) AS total_customers,
    ROUND(AVG(churn_score), 2) AS avg_churn_score
FROM customer_churn_scores
GROUP BY risk_level
ORDER BY avg_churn_score DESC;
'''

churn_summary = pd.read_sql_query(query, conn)
churn_summary


# 20. Business Question 6: Top High-Risk Customers


In [ ]:
query = '''
SELECT 
    c.customer_id,
    c.full_name,
    c.city,
    c.customer_segment,
    ch.churn_score,
    ch.risk_level,
    ch.main_risk_reason,
    ch.recommended_action
FROM customer_churn_scores ch
JOIN customers c
    ON ch.customer_id = c.customer_id
WHERE ch.risk_level = 'High'
ORDER BY ch.churn_score DESC
LIMIT 10;
'''

high_risk_customers = pd.read_sql_query(query, conn)
high_risk_customers


# 21. Business Question 7: Customer Value Segments

Question:

**Which customer segments bring the most revenue?**


In [ ]:
query = '''
SELECT 
    value_segment,
    COUNT(*) AS total_customers,
    ROUND(AVG(arpu_jod), 2) AS avg_arpu,
    ROUND(AVG(total_revenue_6m_jod), 2) AS avg_revenue_6m
FROM customer_value_segments
GROUP BY value_segment
ORDER BY avg_revenue_6m DESC;
'''

value_segments = pd.read_sql_query(query, conn)
value_segments


# 22. Powerful Business Question: High Value + High Churn Risk

Question:

**Which high-value customers are also at high risk of churn?**

This query can later become the foundation for a **Churn Rescue AI Agent**.


In [ ]:
query = '''
SELECT 
    c.customer_id,
    c.full_name,
    c.city,
    c.customer_segment,
    v.value_segment,
    v.arpu_jod,
    v.total_revenue_6m_jod,
    ch.churn_score,
    ch.risk_level,
    ch.main_risk_reason,
    ch.recommended_action
FROM customers c
JOIN customer_churn_scores ch
    ON c.customer_id = ch.customer_id
JOIN customer_value_segments v
    ON c.customer_id = v.customer_id
WHERE ch.risk_level = 'High'
ORDER BY v.total_revenue_6m_jod DESC
LIMIT 10;
'''

vip_churn_risk = pd.read_sql_query(query, conn)
vip_churn_risk


# 23. Business Question 8: Complaint Analysis

Question:

**What are customers complaining about?**


In [ ]:
query = '''
SELECT 
    complaint_category,
    severity,
    COUNT(*) AS total_complaints
FROM complaints
GROUP BY complaint_category, severity
ORDER BY total_complaints DESC
LIMIT 15;
'''

complaint_summary = pd.read_sql_query(query, conn)
complaint_summary


# 24. Business Question 9: Support Interaction Analysis

Question:

**Why are customers contacting support?**


In [ ]:
query = '''
SELECT 
    channel,
    reason_category,
    COUNT(*) AS total_interactions
FROM support_interactions
GROUP BY channel, reason_category
ORDER BY total_interactions DESC
LIMIT 20;
'''

support_summary = pd.read_sql_query(query, conn)
support_summary


# 25. Business Question 10: Network Events

Question:

**Which network events affected many customers?**

This can later become a **Network-Aware Service Recovery Agent**.


In [ ]:
query = '''
SELECT 
    nt.city,
    nt.technology,
    ne.event_type,
    ne.severity,
    COUNT(*) AS total_events,
    SUM(ne.affected_customers) AS total_affected_customers
FROM network_events ne
JOIN network_towers nt
    ON ne.tower_id = nt.tower_id
GROUP BY nt.city, nt.technology, ne.event_type, ne.severity
ORDER BY total_affected_customers DESC
LIMIT 15;
'''

network_summary = pd.read_sql_query(query, conn)
network_summary


# 26. Business Question 11: Campaign Performance

Question:

**Which campaigns worked best?**


In [ ]:
query = '''
SELECT 
    ca.campaign_name,
    ca.campaign_type,
    ca.target_segment,
    COUNT(cr.response_id) AS total_sent,
    SUM(cr.converted_flag) AS total_converted,
    ROUND(100.0 * SUM(cr.converted_flag) / COUNT(cr.response_id), 2) AS conversion_rate_percent
FROM campaigns ca
JOIN customer_campaign_responses cr
    ON ca.campaign_id = cr.campaign_id
GROUP BY ca.campaign_id, ca.campaign_name, ca.campaign_type, ca.target_segment
ORDER BY conversion_rate_percent DESC;
'''

campaign_performance = pd.read_sql_query(query, conn)
campaign_performance


# 27. Create Simple Python Functions

Now we prepare for LangChain tools.

A LangChain tool is often just a Python function that the AI agent can call.

So before we use LangChain, let us create normal Python functions.


## Function 1: Get Customer Profile


In [ ]:
def get_customer_profile(customer_id):
    query = '''
    SELECT 
        c.customer_id,
        c.full_name,
        c.gender,
        c.age_group,
        c.city,
        c.governorate,
        c.customer_segment,
        c.preferred_language,
        c.status,
        a.account_type,
        a.account_status
    FROM customers c
    LEFT JOIN accounts a
        ON c.customer_id = a.customer_id
    WHERE c.customer_id = ?;
    '''

    return pd.read_sql_query(query, conn, params=(customer_id,))


In [ ]:
get_customer_profile(42)


## Function 2: Get Customer Plan


In [ ]:
def get_customer_plan(customer_id):
    query = '''
    SELECT 
        c.customer_id,
        c.full_name,
        s.subscription_id,
        s.msisdn,
        s.service_type,
        s.status AS subscription_status,
        p.plan_name,
        p.plan_category,
        p.monthly_fee_jod,
        p.data_allowance_gb,
        p.local_minutes,
        p.contract_months
    FROM customers c
    JOIN subscriptions s
        ON c.customer_id = s.customer_id
    JOIN plans p
        ON s.plan_id = p.plan_id
    WHERE c.customer_id = ?;
    '''

    return pd.read_sql_query(query, conn, params=(customer_id,))


In [ ]:
get_customer_plan(42)


## Function 3: Get Customer Churn Risk


In [ ]:
def get_customer_churn_risk(customer_id):
    query = '''
    SELECT 
        c.customer_id,
        c.full_name,
        ch.score_month,
        ch.churn_score,
        ch.risk_level,
        ch.main_risk_reason,
        ch.recommended_action
    FROM customer_churn_scores ch
    JOIN customers c
        ON ch.customer_id = c.customer_id
    WHERE c.customer_id = ?;
    '''

    return pd.read_sql_query(query, conn, params=(customer_id,))


In [ ]:
get_customer_churn_risk(42)


## Function 4: Get Recent Complaints


In [ ]:
def get_customer_complaints(customer_id, limit=5):
    query = '''
    SELECT 
        complaint_date,
        complaint_category,
        complaint_description,
        severity,
        status,
        resolution_summary
    FROM complaints
    WHERE customer_id = ?
    ORDER BY complaint_date DESC
    LIMIT ?;
    '''

    return pd.read_sql_query(query, conn, params=(customer_id, limit))


In [ ]:
get_customer_complaints(42)


## Function 5: Get Support Interactions


In [ ]:
def get_customer_support_interactions(customer_id, limit=5):
    query = '''
    SELECT 
        interaction_datetime,
        channel,
        reason_category,
        issue_type,
        priority,
        status,
        sentiment,
        resolution_summary
    FROM support_interactions
    WHERE customer_id = ?
    ORDER BY interaction_datetime DESC
    LIMIT ?;
    '''

    return pd.read_sql_query(query, conn, params=(customer_id, limit))


In [ ]:
get_customer_support_interactions(42)


# 28. Build a Simple Customer 360 Function

This function combines:

- Profile
- Plan
- Churn risk
- Complaints
- Support interactions

This becomes the foundation for a future LangChain tool and MCP tool.


In [ ]:
def get_customer_360(customer_id):
    profile = get_customer_profile(customer_id)
    plans = get_customer_plan(customer_id)
    churn = get_customer_churn_risk(customer_id)
    complaints = get_customer_complaints(customer_id)
    support = get_customer_support_interactions(customer_id)

    return {
        "profile": profile,
        "plans": plans,
        "churn": churn,
        "complaints": complaints,
        "support": support
    }


In [ ]:
customer_360 = get_customer_360(42)

customer_360["profile"]


In [ ]:
customer_360["plans"]


In [ ]:
customer_360["churn"]


In [ ]:
customer_360["complaints"]


In [ ]:
customer_360["support"]


# 29. Convert Customer 360 into Text

This prepares us for LLMs, RAG, and agents.

Structured database records can be converted into text context for an LLM.


In [ ]:
def customer_360_to_text(customer_id):
    data = get_customer_360(customer_id)

    text = f'''
Customer 360 Summary for Customer ID: {customer_id}

PROFILE:
{data['profile'].to_string(index=False)}

PLANS:
{data['plans'].to_string(index=False)}

CHURN RISK:
{data['churn'].to_string(index=False)}

RECENT COMPLAINTS:
{data['complaints'].to_string(index=False)}

RECENT SUPPORT INTERACTIONS:
{data['support'].to_string(index=False)}
'''
    return text


In [ ]:
print(customer_360_to_text(42))


# 30. Simple Rule-Based Recommendation

This is not yet an LLM.

This is a simple business logic function.

Later, we can give this information to an AI model and ask it to write a better recommendation.


In [ ]:
def simple_customer_recommendation(customer_id):
    churn_df = get_customer_churn_risk(customer_id)
    complaints_df = get_customer_complaints(customer_id)
    support_df = get_customer_support_interactions(customer_id)

    if churn_df.empty:
        return "No churn data found for this customer."

    risk_level = churn_df.iloc[0]["risk_level"]
    risk_reason = churn_df.iloc[0]["main_risk_reason"]
    recommended_action = churn_df.iloc[0]["recommended_action"]

    number_of_complaints = len(complaints_df)
    number_of_support_cases = len(support_df)

    summary = f'''
Customer ID: {customer_id}

Risk Level: {risk_level}
Main Risk Reason: {risk_reason}
Existing Recommended Action: {recommended_action}

Recent Complaints Found: {number_of_complaints}
Recent Support Interactions Found: {number_of_support_cases}

Suggested Business Action:
'''

    if risk_level == "High":
        summary += "This customer should be prioritized for proactive retention outreach."
    elif risk_level == "Medium":
        summary += "This customer should be monitored and may need a targeted offer."
    else:
        summary += "This customer appears relatively stable but can be considered for loyalty engagement."

    return summary


In [ ]:
print(simple_customer_recommendation(42))


# 31. Exercise 1

Show the top 10 cities by number of customers.


In [ ]:
query = '''
SELECT 
    city,
    COUNT(*) AS total_customers
FROM customers
GROUP BY city
ORDER BY total_customers DESC
LIMIT 10;
'''

pd.read_sql_query(query, conn)


# 32. Exercise 2

Find the top 10 customers with the highest churn score.


In [ ]:
query = '''
SELECT 
    c.customer_id,
    c.full_name,
    c.city,
    ch.churn_score,
    ch.risk_level,
    ch.main_risk_reason
FROM customers c
JOIN customer_churn_scores ch
    ON c.customer_id = ch.customer_id
ORDER BY ch.churn_score DESC
LIMIT 10;
'''

pd.read_sql_query(query, conn)


# 33. Exercise 3

Find complaint categories with the highest number of complaints.


In [ ]:
query = '''
SELECT 
    complaint_category,
    COUNT(*) AS total_complaints
FROM complaints
GROUP BY complaint_category
ORDER BY total_complaints DESC;
'''

pd.read_sql_query(query, conn)


# 34. Exercise 4

Create a function called `get_customer_billing_summary(customer_id)`.


In [ ]:
def get_customer_billing_summary(customer_id):
    query = '''
    SELECT 
        c.customer_id,
        c.full_name,
        i.invoice_id,
        i.issue_date,
        i.due_date,
        i.total_amount_jod,
        i.amount_due_jod,
        i.invoice_status
    FROM customers c
    JOIN accounts a
        ON c.customer_id = a.customer_id
    JOIN invoices i
        ON a.account_id = i.account_id
    WHERE c.customer_id = ?
    ORDER BY i.issue_date DESC
    LIMIT 5;
    '''

    return pd.read_sql_query(query, conn, params=(customer_id,))


In [ ]:
get_customer_billing_summary(42)


# 35. Capstone Project Preview

In the final workshop session, teams can build one of these projects using the same database.

## Project Option 1: Customer Care AI Copilot

Input: Customer ID  
Output: profile, plan, complaints, churn risk, and recommended action

## Project Option 2: Churn Rescue Assistant

Input: risk level or segment  
Output: high-risk customers and retention recommendations

## Project Option 3: Billing Support Assistant

Input: customer ID  
Output: invoice summary, payment status, and possible issue

## Project Option 4: Network Impact Assistant

Input: city or tower  
Output: network events and affected customer groups

## Project Option 5: Campaign Recommendation Assistant

Input: customer segment  
Output: best campaign or offer

## Project Option 6: Plan Recommendation Assistant

Input: customer usage pattern  
Output: better plan or add-on


# 36. How This Connects to LangChain and MCP

Today we created normal Python functions such as:

- `get_customer_profile`
- `get_customer_plan`
- `get_customer_churn_risk`
- `get_customer_complaints`
- `get_customer_support_interactions`
- `get_customer_360`

In the next sessions:

1. We will convert these functions into LangChain tools.
2. We will build a LangChain SQL Agent.
3. We will create simple RAG from database rows.
4. We will create a simple multi-agent customer care copilot.
5. We will expose selected functions as MCP tools.

MCP will allow our AI application to expose tools in a standard way, so agents can call them through a common protocol.


# 37. Trainer Closing Script

Today, we completed the foundation.

We connected Python to the Zain Jordan Customer 360 database, explored telecom data, asked business questions using SQL, and created reusable Python functions.

These functions are not just Python practice. They are the building blocks for AI tools.

In the next session, we will convert this into LangChain tools and agents. Then we will move toward RAG, multi-agent workflows, and MCP.

By the final day, teams will use this same database to build and present a capstone AI project.
